# Downloading LOONE Run Data

This notebook shows how to use python to download the files associated with the LOONE runs carried out by the [Loone Forecast App](https://close-habs.aquaveo.com/apps/loone-model/)

## Using Python

#### 1. Install google-cloud-storage Python Package

Python can be used to download the LOONE data by using the `google-cloud-storage` python package. First, we need to install `google-cloud-storage` using `pip`:

In [ ]:
# Install google-cloud-storage python package. Skip if already installed
%pip install google-cloud-storage

> Running `%pip install google-cloud-storage` in the cell above installs the `google-cloud-storage` package into the current notebook kernel.
>
> If you want to install `google-cloud-storage` from a terminal instead, run `pip install google-cloud-storage` in the same Python environment used by this notebook.

#### 2. Import Packages

In [ ]:
# Imports
from datetime import datetime
import os
from google.cloud import storage

#### 3. Helper Functions

The following function can be used to list which dates have data available:

In [ ]:
def print_dates_with_archived_data():
    """Print the dates for which archived data is available."""
    # Get client and bucket for downloading the data
    bucket_name = 'loone-forecast-datastore'
    prefix = f'public'
    client = storage.Client.create_anonymous_client()
    
    # List blobs in the bucket with the specified prefix
    blobs = client.list_blobs(bucket_name, prefix=prefix)
    
    # Extract unique dates from the blob names
    dates = set()
    
    for blob in blobs:
        # Extract the date from the blob name
        date = blob.name.split('/')[1]

        try:
            # Check that the current item is a date prefix
            if datetime.strptime(date, '%Y-%m-%d'):
                # Current item is a date prefix, add it to the set of dates
                dates.add(date)
        except ValueError:
            # Current item is not a date prefix, skip it
            continue
    
    # Print the sorted list of unique dates
    for date in sorted(dates):
        print(date)

The following function can be used to download the archived data of the [Loone Forecast App](https://close-habs.aquaveo.com/apps/loone-model/):

In [ ]:
def download_archived_data(date, output_path: str):
    """Downloads the archived data for the specified date to the given output location.

    Args:
       (datetime.datetime) date: The date of the run you want the data from.
       (str) output_path: The path to the directory where the downloaded files should go.
    """
    # Get date as a string in the expected format
    date_string = date.strftime('%Y-%m-%d')
    
    # Get client and bucket for downloading the data
    bucket_name = 'loone-forecast-datastore'
    prefix = f'public/{date_string}'
    client = storage.Client.create_anonymous_client()
    bucket = client.bucket(bucket_name)

    # Download all the files for the given date
    file_download_count = 0

    try:
        for blob in client.list_blobs(bucket_name, prefix=prefix):
            # Get the location the file will be downloaded to
            blob_output_path = os.path.join(output_path, blob.name.removeprefix('public/'))
            blob_output_dir = os.path.dirname(blob_output_path)
    
            # Create the directories the files will be output to
            if not os.path.exists(blob_output_dir):
                os.makedirs(blob_output_dir, exist_ok=True)
            
            # Download the file
            blob.download_to_filename(blob_output_path)
    
            # Notify user of progress
            print(f'Downloaded {blob.name}')
    
            # Keep track of how many files we download
            file_download_count += 1
    except Exception as e:
        print(f'Failed to download {blob.name}')
    
    # Notify user the downloads are complete
    print(f'Downloads Complete! {file_download_count} file(s) downloaded')

#### 4. Run the Functions

##### Print the dates that have data available:

In [ ]:
print_dates_with_archived_data()

##### Download Data for a Specified Date:

In [ ]:
# Configuration
date = datetime(2026, 4, 10)                                # The date of the run whose data you want
output_path = os.path.join(os.getcwd(), 'archived-data')    # The directory where you want the downloaded files to go

# Download the archived data for the specified date to the given output location
download_archived_data(date, output_path)